# ChromaDB with free local Ollama embeddings

Prerequisites: install Ollama, run `ollama pull nomic-embed-text`, and make sure the Ollama service is running. This notebook uses local embeddings and does not need an OpenAI API key.

A separate `my_chroma_db_ollama` directory is used because vectors generated by different embedding models should not be mixed.


In [1]:
# No API key required: embeddings will run locally through Ollama.


In [7]:
!pip install -U langchain-ollama langchain-chroma chromadb


Defaulting to user installation because normal site-packages is not writeable
  Using cached websocket_client-1.9.2-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ------------- -------------------------- 8.1/23.5 MB 43.1 MB/s eta 0:00:01
   -------------------------------- ------- 19.1/23.5 MB 47.3 MB/s eta 0:00:01
   ---------------------------------------- 23.5/23.5 MB 45.0 MB/s  0:00:00
   ---------------------------------------- 0.0/7.5 MB ? eta -:--:--
   ---------------------------------------- 7.5/7.5 MB 46.9 MB/s  0:00:00
   ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
   ------------------------------ --------- 11.5/15.3 MB 53.4 MB/s eta 0:00:01
   ---------------------------------------- 15.3/15.3 MB 44.8 MB/s  0:00:00
Using cached websocket_client-1.9.2-py3-none-any.whl (95 kB)

   ---- -----------------------------------  3/26 [websocket-client]
   ---------- -----------------------------  7/

In [8]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document


In [9]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [10]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [12]:
vector_store = Chroma(
    embedding_function=OllamaEmbeddings(model="bge-m3"),
    persist_directory='my_chroma_db_ollama',
    collection_name='sample'
)

In [13]:
vector_store.add_documents(docs, ids=['kohli', 'rohit', 'dhoni', 'bumrah', 'jadeja'])

['kohli', 'rohit', 'dhoni', 'bumrah', 'jadeja']

In [14]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['kohli', 'rohit', 'dhoni', 'bumrah', 'jadeja'],
 'embeddings': array([[-0.02092008,  0.01448835, -0.02798354, ..., -0.00116182,
          0.01595093, -0.05414918],
        [ 0.02540159, -0.00023502, -0.02140886, ..., -0.01172258,
          0.03286374, -0.06022512],
        [ 0.00521872,  0.00041571, -0.03275013, ..., -0.01750591,
          0.04059405, -0.04632686],
        [-0.00783331,  0.00621713, -0.04659764, ..., -0.06779984,
          0.00766559, -0.03191799],
        [-0.01421967,  0.02873416, -0.0541212 , ..., -0.01257669,
         -0.01086697, -0.01217784]], shape=(5, 1024)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",


In [15]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='bumrah', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='kohli', metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.')]

In [16]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='bumrah', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9790631532669067),
 (Document(id='kohli', metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.'),
  1.0077286958694458)]

In [17]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='dhoni', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.0344089269638062),
 (Document(id='jadeja', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.1154451370239258)]

In [18]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='kohli', document=updated_doc1)


In [19]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['kohli', 'rohit', 'dhoni', 'bumrah', 'jadeja'],
 'embeddings': array([[-0.01380858,  0.0165603 , -0.03242373, ..., -0.00364471,
          0.07846843, -0.03323471],
        [ 0.02540159, -0.00023502, -0.02140886, ..., -0.01172258,
          0.03286374, -0.06022512],
        [ 0.00521872,  0.00041571, -0.03275013, ..., -0.01750591,
          0.04059405, -0.04632686],
        [-0.00783331,  0.00621713, -0.04659764, ..., -0.06779984,
          0.00766559, -0.03191799],
        [-0.01421967,  0.02873416, -0.0541212 , ..., -0.01257669,
         -0.01086697, -0.01217784]], shape=(5, 1024)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to

In [20]:
# delete document
vector_store.delete(ids=['kohli'])

In [21]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['rohit', 'dhoni', 'bumrah', 'jadeja'],
 'embeddings': array([[ 0.02540159, -0.00023502, -0.02140886, ..., -0.01172258,
          0.03286374, -0.06022512],
        [ 0.00521872,  0.00041571, -0.03275013, ..., -0.01750591,
          0.04059405, -0.04632686],
        [-0.00783331,  0.00621713, -0.04659764, ..., -0.06779984,
          0.00766559, -0.03191799],
        [-0.01421967,  0.02873416, -0.0541212 , ..., -0.01257669,
         -0.01086697, -0.01217784]], shape=(4, 1024)),
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over